In [ ]:
# VIIRS ENV 모드

In [1]:
import os
from glob import glob
import rasterio
import fiona
from rasterio.features import geometry_mask

import pandas as pd
import geopandas as gpd

from tqdm import tqdm

In [34]:
#set default directory
os.chdir(r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\imagery_data_complete_0828\data')

image_path = 'image'

shp_path = '../../Fire_boundary_polygon_files/0911_ver/2016_2022.shp'

In [49]:
output_path = 'mask'

In [35]:
shp = gpd.read_file(shp_path)

In [36]:
shp

,Fire_masks,Fire_IDs,I_date,Sub_ID,Condition,ID,geometry
0,1,2016001,2016-01-26,1,1,1,"POLYGON ((492879.709 4172479.976, 492890.292 4..."
1,1,2016002,2016-02-15,1,1,2,"POLYGON ((470639.734 4091620.085, 470639.734 4..."
2,1,2016003,2016-03-09,1,1,3,"POLYGON ((415080.001 4146020.205, 415090.655 4..."
3,1,2016003,2016-02-18,1,4,4,"POLYGON ((415080.001 4146020.205, 415090.655 4..."
4,1,2016004,2016-03-09,1,1,5,"MULTIPOLYGON (((346249.744 4146730.485, 346249..."
...,...,...,...,...,...,...,...
348,1,2022025,2022-03-03,1,1,286,"POLYGON ((440819.748 4200770.538, 440819.962 4..."
349,1,2022052,2022-04-09,1,2,316,"MULTIPOLYGON (((469969.508 4084510.549, 469970..."
350,1,2022052,2022-04-12,1,1,317,"MULTIPOLYGON (((469959.508 4084520.549, 469960..."
351,1,2019029,2019-04-13,1,4,178,"MULTIPOLYGON (((427300.182 4208350.060, 427300..."


In [38]:
# list satellite images using glob (list)
images = glob(f'{image_path}/*.tif')


In [39]:
len(images)

353

In [51]:
images[5]

'image\\T52SBE_20210407T021559_2021017.tif'

In [54]:
images[5][6:-4]

'T52SBE_20210407T021559_2021017'

In [53]:
images[5][13:21]

'20210407'

In [50]:
images[5][-11:-4]

'2021017'

In [19]:
images[5][-11:-4]

'2021017'

In [37]:
shp

,Fire_masks,Fire_IDs,I_date,Sub_ID,Condition,ID,geometry
0,1,2016001,2016-01-26,1,1,1,"POLYGON ((492879.709 4172479.976, 492890.292 4..."
1,1,2016002,2016-02-15,1,1,2,"POLYGON ((470299.338 4091289.964, 470299.338 4..."
2,1,2016003,2016-03-09,1,1,3,"POLYGON ((415080.001 4146020.205, 415090.655 4..."
3,1,2016003,2016-02-18,1,4,4,"POLYGON ((415080.001 4146020.205, 415090.655 4..."
4,1,2016004,2016-03-09,1,1,5,"MULTIPOLYGON (((346249.744 4146730.485, 346249..."
...,...,...,...,...,...,...,...
348,1,2022085,2022-12-30,1,2,353,"POLYGON ((489402.760 3886239.972, 489420.074 3..."
349,1,2022025,2022-03-03,1,1,286,"POLYGON ((440819.748 4200770.538, 440819.962 4..."
350,1,2022052,2022-04-09,1,2,316,"MULTIPOLYGON (((469969.508 4084510.549, 469970..."
351,1,2022052,2022-04-12,1,1,317,"MULTIPOLYGON (((469959.508 4084520.549, 469960..."


In [51]:

#select first image data through loop
for inum in images:
    with rasterio.open(inum) as src:
        #read the image as numpy array
        image = src.read()
        
        Image_FID = int(inum[-11:-4])
        Image_Date = int(inum[39:47])     
        
        
#         for snum in shps:
             
#             #shp_range_text = snum[-19:-4]
#             shp_s_range = int(snum[-19:-12])
#             shp_e_range = int(snum[-11:-4])
#             shp_id_list = list(range(shp_s_range, shp_e_range+1))
            
#         if Image_FID not in shp_id_list:
#             print(Image_FID)
#             continue
#         else:
        with fiona.open(snum) as shapefile:
            for feature in shapefile:
                if feature['properties']['Fire_IDs']==str(Image_FID) and feature['properties']['I_date'].replace('-','')==str(Image_Date):
                    target_geometry = feature["geometry"]
                            
                    # Generate a mask from the geometry that has the same shape as the image
                    mask = geometry_mask([target_geometry], out_shape=image.shape[1:], transform=src.transform, invert=True)
                            
                    # Save the mask to a new GeoTIFF file
                            
                    output_file = f"{output_path}/{inum[16:-4]}_mask.tif"
                    with rasterio.open(output_file, "w", driver="GTiff", height=image.shape[1], width=image.shape[2], count=1, dtype="uint8", crs=src.crs, transform=src.transform) as dst:
                        dst.write(mask.astype("uint8"), 1)


In [47]:
os.getcwd()

'D:\\deep_learning\\Sentinel_forestfire\\MY_TEST_SENTINEL_IMAGERY_DATA\\imagery_data_complete_0828\\data'

In [48]:
output_path

'imagery_data_complete_0828/mask'

In [55]:
for index, row in tqdm(shp.iterrows(), total = shp.shape[0]):
    Shape_date = row['I_date'].replace('-', '')
    Fire_ID = row['Fire_IDs']
    target_geometry = row['geometry']
    
    #print(Shape_date, '\n', Fire_ID, '\n', target_geometry)
    #break
    for img_n in images:
        Image_ID = img_n[-11:-4]
        Image_date = img_n[13:21]
        

        
        
        
        if Image_date == Shape_date and Image_ID == Fire_ID:
            with rasterio.open(img_n) as src:
                #read the image as numpy array
                image = src.read()
                mask = geometry_mask([target_geometry], out_shape=image.shape[1:], transform=src.transform, invert=True)
                            
                    # Save the mask to a new GeoTIFF file
                            
                output_file = f"{output_path}/{img_n[6:-4]}_mask.tif"
                with rasterio.open(output_file, "w", driver="GTiff", height=image.shape[1], width=image.shape[2], count=1, dtype="uint8", crs=src.crs, transform=src.transform) as dst:
                    dst.write(mask.astype("uint8"), 1)
    

100%|████████████████████████████████████████████████████████████████████████████████| 353/353 [00:07<00:00, 45.73it/s]
